# derived_8.4-eval-mlp-2.0 — optimized MLP architecture to break the ceiling (2.0 H100-hours)

Follow-up to `derived_8.4-eval-mlp-1.3` (plain-2-regime-MLP ceiling confirmed: val-selected winners 2regime_96 0.761 / 2regime_54 0.765, test-best 0.789, val top-10 avg 0.7825; XGBoost 2-regime 0.815) and `derived_8.4-eval-2.0` (LOSO: the 96-pool transfers spatially better — pooled LOSO 0.668 ≈ XGBoost 2-regime 0.689). 2.0 attacks the remaining ceiling with an **optimized architecture + training**:

1. **FeatureGroupedMLP (`fg`)** — per-semantic-group towers + fusion MLP (grouping in `mlp20/feature_groups.py`, validated: every feature in exactly one group). Targets 1.2's documented overfitting kind — capacity spent on period-specific *interactions* between heterogeneous sensor families.
2. **PLRRegressor (`plr`)** — piecewise-linear encoding (Gorishniy et al. 2022) + plain-MLP body.
3. **SWA (trainer)** — Stochastic Weight Averaging updated once per epoch with BN recalibration; replaces the 1.3 per-step EMA (documented trainer failure). Deployed iff its best val RMSE beats the live best (honest within-val comparison).
4. **New family `2regime_mixed`** — c0 = 96-pool, c1 = 54+10 deltas: the per-cluster-optimal feature allocation (mlp-1.3 per-cluster R²: c0-96 0.754 > c0-54 0.737; c1-54 0.831 > c1-96 0.776).

Protocol unchanged and honest: train on train (2017–2020, n=9,803), early-stop/select on the official val (2021–2022, n=4,805), evaluate on the untouched test (2023–2025, n=6,620); 2-seed mean val RMSE selection (mlp/fg/plr only — residual/ft are reference-only); aux2020 diagnostic only; patience-60 kept (1.3-confirmed); no calibration / no trainval retrain (documented negatives). LOSO is explicitly out of the sweep (per the experiment brief).

All numbers below are the stdout of this executed notebook. Weights/checkpoints/test predictions under `models/`; preprocessed tensors and per-job logs under `artifacts/`; figures at the experiment root.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Robust resolution of the experiment dir whether executed from notebooks/ or in-place.
candidates = [Path.cwd() / "experiment/derived_8.4-eval-mlp-2.0", Path.cwd()]
EXP_DIR = next((p for p in candidates if (p / "metrics_summary.csv").exists()), Path.cwd())

df_summary = pd.read_csv(EXP_DIR / "metrics_summary.csv")
df_per_regime = pd.read_csv(EXP_DIR / "per_regime_metrics_summary.csv")
df_sweep = pd.read_csv(EXP_DIR / "sweep_results.csv")
df_timing = pd.read_csv(EXP_DIR / "timing_summary.csv")
df_bias = pd.read_csv(EXP_DIR / "bias_summary.csv") if (EXP_DIR / "bias_summary.csv").exists() else None
df_bias_cl = pd.read_csv(EXP_DIR / "bias_by_cluster.csv") if (EXP_DIR / "bias_by_cluster.csv").exists() else None
df_ood = pd.read_csv(EXP_DIR / "ood_summary.csv") if (EXP_DIR / "ood_summary.csv").exists() else None
df_stop20 = pd.read_csv(EXP_DIR / "stopping_20_summary.csv") if (EXP_DIR / "stopping_20_summary.csv").exists() else None
df_stop20_agg = pd.read_csv(EXP_DIR / "stopping_20_aggregate.csv") if (EXP_DIR / "stopping_20_aggregate.csv").exists() else None
with open(EXP_DIR / "selected_features.json") as f:
    selected_meta = json.load(f)
with open(EXP_DIR / "timing_log.json") as f:
    timing_log = json.load(f)

fam_labels = {"2regime_96": "2-Regime-96", "2regime_54": "2-Regime-54", "2regime_mixed": "2-Regime-Mixed"}
print("loaded", len(df_summary), "leaderboard rows,", len(df_sweep), "sweep rows")


loaded 45 leaderboard rows, 30 sweep rows


## Selection Protocol v6 Diagnostic

1.2 documented that aux2020 (2020 ⊂ train) measures train fit, not generalization — it stays diagnostic-only. 1.3 confirmed no honest early-stopping rule beats patience-60 and that the val ranking is the honest signal, stabilized by 2-seed averaging. 2.0 selects by **2-seed mean val RMSE** among the honest architectures (mlp / fg / plr; residual/ft are reference-only). This section reports the val ranking, the Spearman correlations vs test, and the per-architecture winner so the selection is auditable (no test-based cherry-picking).


In [2]:
from scipy.stats import spearmanr
HONEST = ("mlp", "fg", "plr")
print("### Selection Protocol v6 Diagnostic (selection = 2-seed mean val RMSE; mlp/fg/plr only)")
for family, fam_label in fam_labels.items():
    sub = df_sweep[df_sweep["family"] == family].dropna(subset=["test_r2"]).copy()
    if sub.empty:
        continue
    sub = sub.sort_values("val_rmse", na_position="last").reset_index(drop=True)
    print(f"\n#### {fam_label} — top-10 by val RMSE")
    cols = ["config_id", "architecture", "n_seeds", "val_rmse", "aux_rmse", "test_r2", "test_rmse", "test_bias"]
    print(sub.head(10)[cols].to_markdown(index=False))
    for metric, label in [("val_rmse", "val_rmse"), ("robust_score", "robust_score"), ("aux_rmse", "aux_rmse")]:
        valid = sub.dropna(subset=[metric, "test_r2"])
        if len(valid) >= 8:
            rho, p = spearmanr(valid[metric], valid["test_r2"])
            print(f"  Spearman({label}, test_r2) = {rho:+.3f} (p={p:.3f}, n={len(valid)})")
    honest_sub = sub[sub["architecture"].isin(HONEST)]
    val_honest = honest_sub.sort_values("val_rmse").iloc[0]
    test_best = sub.sort_values("test_r2", ascending=False).iloc[0]
    print(f"  val winner (honest) : {val_honest['config_id']} (test_r2={val_honest['test_r2']:.4f})")
    print(f"  test best (ref)     : {test_best['config_id']} (test_r2={test_best['test_r2']:.4f})")


### Selection Protocol v6 Diagnostic (selection = 2-seed mean val RMSE; mlp/fg/plr only)

#### 2-Regime-96 — top-10 by val RMSE
| config_id                      | architecture   |   n_seeds |   val_rmse |   aux_rmse |   test_r2 |   test_rmse |   test_bias |
|:-------------------------------|:---------------|----------:|-----------:|-----------:|----------:|------------:|------------:|
| w512x512x512_d0.3_lr1e-3       | mlp            |         2 |  0.0482834 |  0.0249457 |  0.761018 |   0.0497987 |  0.0176019  |
| w512x512x512_d0.3_lr1e-3_swa   | mlp            |         2 |  0.0483296 |  0.0249357 |  0.754698 |   0.050453  |  0.0204438  |
| w512x512x512_d0.3_huber0.1_swa | mlp            |         2 |  0.0490918 |  0.0258471 |  0.773153 |   0.048518  |  0.0163212  |
| fg_w384x384_d0.3_huber0.1      | fg             |         2 |  0.0508182 |  0.0305982 |  0.710904 |   0.0547718 |  0.0220846  |
| fg_w384x384_d0.3_huber0.1_swa  | fg             |         2 |  0.0509411 |  0.0312027 |  0

## Overall Model Leaderboard

All evaluated models ranked by pooled test R² over 2023–2025 (6,620 samples, 7 WA stations). MLP rows carry the sweep `config_id` and `n_seeds`; `(val top-k avg)` rows are offline seed-averaged ensembles of the top-k val-selected honest configs (no extra training); `(5-seed champ, ...)` rows are 5-seed champion ensembles of the val-selected winners (extra stability seeds, no trainval retrain — documented negative); `cross-family` rows average the val-selected winners across families (54/96/mixed are complementary: near-unbiased / extrapolation / per-cluster-optimal). XGBoost rows are the eval-1.1 references; `MLP-1.3` rows are the 1.3 val-selected winners + test-best references; `test-best` rows are reporting only (selection on test would be leakage).


In [3]:
cols = ["model_name", "strategy_name", "pooled_r2", "pooled_rmse", "pooled_ubrmse", "pooled_bias", "pooled_mae", "pooled_pearson"]
print("### Overall Leaderboard (2023-2025 Test Set)")
print(df_summary[cols].to_markdown(index=False))


### Overall Leaderboard (2023-2025 Test Set)
| model_name                                                                                                                                                | strategy_name          |   pooled_r2 |   pooled_rmse |   pooled_ubrmse |   pooled_bias |   pooled_mae |   pooled_pearson |
|:----------------------------------------------------------------------------------------------------------------------------------------------------------|:-----------------------|------------:|--------------:|----------------:|--------------:|-------------:|-----------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10)                                                                                                                | XGBoost_Reference      |    0.81496  |     0.0438196 |       0.043337  |   0.00648567  |    0.0337195 |         0.905594 |
| MLP 2-Regime-Mixed (val top-5 avg)                                                                            

## Hyperparameter Sweep Summary

30 curated configs × 3 families (2regime_54, 2regime_96, NEW 2regime_mixed) — 1.3 anchors re-run under the v6 protocol, SWA variants, mixup α=0.4, grouped-tower (`fg`) and PLR (`plr`) architectures, and bias-targeting small nets for the 96-family. 8 parallel H100 workers; configs ranked by **2-seed mean val RMSE** (honest signal); test R² for reference. Phase-2 configs carry `n_seeds=2`.


In [4]:
for family, fam_label in fam_labels.items():
    sub = df_sweep[df_sweep["family"] == family].sort_values("val_rmse", na_position="last").head(10)
    print(f"### Sweep Top-10 — {fam_label} (by val RMSE, the honest selection signal)")
    cols = ["config_id", "architecture", "n_seeds", "dropout", "lr", "loss", "val_rmse", "aux_rmse", "test_r2", "test_rmse", "test_bias", "best_epoch", "train_time_s"]
    show = sub[cols].copy()
    show["deployed"] = [json.loads((EXP_DIR / "models" / family / cid / "meta.json").read_text()).get("deployed", "live")
                        if (EXP_DIR / "models" / family / cid / "meta.json").exists() else "" for cid in sub["config_id"]]
    print(show.to_markdown(index=False))
    print()


### Sweep Top-10 — 2-Regime-96 (by val RMSE, the honest selection signal)
| config_id                      | architecture   |   n_seeds |   dropout |     lr | loss   |   val_rmse |   aux_rmse |   test_r2 |   test_rmse |   test_bias |   best_epoch |   train_time_s | deployed   |
|:-------------------------------|:---------------|----------:|----------:|-------:|:-------|-----------:|-----------:|----------:|------------:|------------:|-------------:|---------------:|:-----------|
| w512x512x512_d0.3_lr1e-3       | mlp            |         2 |      0.3  | 0.001  | mse    |  0.0482834 |  0.0249457 |  0.761018 |   0.0497987 |  0.0176019  |          263 |       122.396  | live       |
| w512x512x512_d0.3_lr1e-3_swa   | mlp            |         2 |      0.3  | 0.001  | mse    |  0.0483296 |  0.0249357 |  0.754698 |   0.050453  |  0.0204438  |          266 |       123.883  | live       |
| w512x512x512_d0.3_huber0.1_swa | mlp            |         2 |      0.3  | 0.0003 | huber  |  0.0490918 |

## Per-Regime Performance Breakdown

Cluster 0 holds 73% of the test rows, so it dominates the pooled R². Per-cluster test metrics for the val top-3 honest configs per family (including the new `2regime_mixed` family, whose c1 = 54+10 specialist is expected to hold the ~0.83 R² of the 54-family's c1 while c0 gains the 96-pool), the XGBoost references, and the 1.3 reference winners.


In [5]:
print("### Per-Regime Performance Breakdown")
cols = ["strategy_name", "model_name", "cluster", "n_train", "n_test", "r2", "rmse", "ubrmse", "bias", "mae"]
print(df_per_regime[cols].to_markdown(index=False))


### Per-Regime Performance Breakdown
| strategy_name     | model_name                                          |   cluster |   n_train |   n_test |       r2 |      rmse |    ubrmse |         bias |       mae |
|:------------------|:----------------------------------------------------|----------:|----------:|---------:|---------:|----------:|----------:|-------------:|----------:|
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_lr1e-3)          |         0 |      7156 |     4817 | 0.754287 | 0.0495899 | 0.0472413 |  0.0150802   | 0.0389389 |
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_lr1e-3)          |         1 |      2647 |     1803 | 0.776352 | 0.0503523 | 0.0440792 |  0.0243389   | 0.0370033 |
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_lr1e-3_swa)      |         0 |      7156 |     4817 | 0.745281 | 0.0504906 | 0.046785  |  0.0189859   | 0.0398074 |
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_lr1e-3_swa)      |         1 |      2647 

## Yearly Performance Breakdown

Year-by-year R² on the 2023–2025 test period — 1.0's analysis showed MLPs degrade most in 2025 (distribution drift); this table tracks whether the 2.0 architecture/regularizers close that year specifically.


In [6]:
year_cols = [c for c in df_summary.columns if c.startswith("year_") and c.endswith("_r2")]
print("### Year-by-Year R² Breakdown")
print(df_summary[["model_name", "pooled_r2", *year_cols]].to_markdown(index=False))


### Year-by-Year R² Breakdown
| model_name                                                                                                                                                |   pooled_r2 |   year_2023_r2 |   year_2024_r2 |   year_2025_r2 |
|:----------------------------------------------------------------------------------------------------------------------------------------------------------|------------:|---------------:|---------------:|---------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10)                                                                                                                |    0.81496  |       0.822971 |       0.783256 |       0.83029  |
| MLP 2-Regime-Mixed (val top-5 avg)                                                                                                                        |    0.800323 |       0.745352 |       0.825612 |       0.832424 |
| MLP 2-Regime-Mixed (val top-3 avg)                                          

## Systematic-Bias Diagnostic (headline)

mlp-1.2/1.3 documented the 96-family's systematic positive test bias (bias² ≈ 10–17% of MSE) and that post-hoc val-fit calibration does NOT transfer. The 2.0 debias levers are architectural + train-time (capacity control, SWA, mixup, grouped towers). This table reports `bias²/MSE` per config (MSE = bias² + ubRMSE²) — the success criterion is a per-family median < 5%. Backed by `analyze_bias.py`.


In [7]:
if df_bias is not None:
    print("### Per-family median bias^2/MSE share (honest architectures)")
    print("| family     | n_configs |   med_bias2_mse_share |   med_test_bias |   med_test_r2 |")
    print("|:-----------|----------:|----------------------:|----------------:|--------------:|")
    for fam, fam_label in fam_labels.items():
        sub = df_bias[(df_bias["family"] == fam) & df_bias["architecture"].isin(HONEST)]
        if sub.empty:
            continue
        print(f"| {fam} | {len(sub)} | {sub['bias2_mse_share'].median():.4f} | "
              f"{sub['test_bias'].median():.4f} | {sub['test_r2'].median():.4f} |")
    cols = ["family", "config_id", "architecture", "test_r2", "test_rmse", "test_bias", "bias2_mse_share"]
    print("\n### Worst 8 configs by bias^2/MSE share (all architectures)")
    print(df_bias.sort_values("bias2_mse_share", ascending=False).head(8)[cols].to_markdown(index=False))
    print("\n### Best 8 configs by bias^2/MSE share (all architectures)")
    print(df_bias.sort_values("bias2_mse_share").head(8)[cols].to_markdown(index=False))
    if df_bias_cl is not None:
        print("\n### Per-cluster median bias^2/MSE share (honest architectures)")
        print("| family     | cluster |   med_bias2_mse_share |   med_test_bias |   med_test_r2 |")
        print("|:-----------|--------:|----------------------:|----------------:|--------------:|")
        for fam, fam_label in fam_labels.items():
            for cl in (0, 1):
                sub = df_bias_cl[(df_bias_cl["family"] == fam) & (df_bias_cl["cluster"] == cl)
                                 & df_bias_cl["architecture"].isin(HONEST)]
                if sub.empty:
                    continue
                print(f"| {fam} | {cl} | {sub['bias2_mse_share'].median():.4f} | "
                      f"{sub['test_bias'].median():.4f} | {sub['test_r2'].median():.4f} |")
else:
    print("bias_summary.csv not found — run analyze_bias.py after the sweep.")


### Per-family median bias^2/MSE share (honest architectures)
| family     | n_configs |   med_bias2_mse_share |   med_test_bias |   med_test_r2 |
|:-----------|----------:|----------------------:|----------------:|--------------:|
| 2regime_96 | 10 | 0.1326 | 0.0188 | 0.7579 |
| 2regime_54 | 10 | 0.0370 | 0.0095 | 0.7646 |
| 2regime_mixed | 10 | 0.1001 | 0.0151 | 0.7702 |

### Worst 8 configs by bias^2/MSE share (all architectures)
| family        | config_id                       | architecture   |   test_r2 |   test_rmse |   test_bias |   bias2_mse_share |
|:--------------|:--------------------------------|:---------------|----------:|------------:|------------:|------------------:|
| 2regime_96    | plr_w256x256_d0.4_swa           | plr            |  0.623881 |   0.0624739 |   0.0340105 |          0.296366 |
| 2regime_mixed | w448x448_d0.3_gelu              | mlp            |  0.760502 |   0.0498525 |   0.0208406 |          0.174762 |
| 2regime_mixed | w448x448_d0.3_gelu_swa       

## SWA vs Live Deployment (trainer)

The mlp20 trainer records for every SWA config, **per specialist and per seed**: the live-model best val RMSE, the SWA-snapshot best val RMSE, and which one was deployed (honest within-val comparison — the SWA snapshot is deployed only if its best val beats the live best). This table reads the per-seed per-cluster metas (the aggregated `meta.json` does not carry these fields).

**Result (documented):** with the 2.0 recipe (`swa_start_frac: 0.6`, equal-weight average over epochs 240–400, BN recalibrated before each SWA-val eval), **no SWA snapshot ever beat the live model on val** (SWA val 0.075–0.12 vs live 0.040–0.053), so every SWA config deployed the **live** best model. The val-based deployment mechanism worked as designed — it correctly rejected non-competitive snapshots.

**Implementation note (reproducibility):** the BN-recalibration pass runs the train loader in train mode, so its dropout layers consume the shared RNG stream; the live trajectory of a `swa=true` job therefore diverges slightly from its `swa=false` anchor (same seed; e.g. mixed `w512x512x512_d0.3_huber0.1_swa` seed 42 best_epoch 266 vs the anchor's 240). The reported swa-config numbers are deterministic and reproducible from the committed code + seeds, but their gains over the anchors are **live-trajectory results, not SWA weight deployment**. A future SWA test should (a) start averaging later (e.g. `swa_start_frac: 0.85`) and (b) guard the RNG around the recalibration pass so the live path stays bit-identical to the anchor.


In [8]:
# Read the per-seed per-cluster metas (the aggregated meta.json does not carry
# the SWA fields; the seed metas are the source of truth).
import json
rows = []
for _, r in df_sweep.iterrows():
    cdir = EXP_DIR / "models" / r["family"] / r["config_id"]
    agg_meta = None
    if (cdir / "meta.json").exists():
        agg_meta = json.loads((cdir / "meta.json").read_text())
    if not agg_meta or not agg_meta.get("config", {}).get("swa", False):
        continue
    # aggregate per-specialist live/SWA best val over the completed seeds
    live_c, swa_c = [], []
    n_deploy_swa = 0
    for s in (42, 7):
        p = cdir / f"seed_{s}" / "meta.json"
        if not p.exists():
            continue
        m = json.loads(p.read_text())
        for cl in ("0", "1"):
            pc = m.get("per_cluster", {}).get(cl, {})
            if pc.get("val_rmse_live") is not None:
                live_c.append(pc["val_rmse_live"])
            if pc.get("val_rmse_swa") is not None and np.isfinite(pc["val_rmse_swa"]):
                swa_c.append(pc["val_rmse_swa"])
            if pc.get("deployed") == "swa":
                n_deploy_swa += 1
    rows.append({
        "family": r["family"], "config_id": r["config_id"],
        "architecture": agg_meta["config"].get("architecture", "mlp"),
        "n_seeds": r.get("n_seeds"),
        "specs_deployed_swa": n_deploy_swa,
        "val_rmse_live": float(np.mean(live_c)) if live_c else float("nan"),
        "val_rmse_swa": float(np.mean(swa_c)) if swa_c else float("nan"),
        "test_r2": r.get("test_r2"), "test_rmse": r.get("test_rmse"), "test_bias": r.get("test_bias"),
    })
swa_df = pd.DataFrame(rows)
if not swa_df.empty:
    print("### SWA configs — per-seed per-cluster live vs SWA val RMSE (mean over specs/seeds)")
    print("specs_deployed_swa = number of (seed, specialist) jobs where the SWA snapshot beat the live best on val.")
    print(swa_df.sort_values(["family", "val_rmse_live"]).to_markdown(index=False))
else:
    print("no SWA configs found in the sweep results")


### SWA configs — per-seed per-cluster live vs SWA val RMSE (mean over specs/seeds)
specs_deployed_swa = number of (seed, specialist) jobs where the SWA snapshot beat the live best on val.
| family        | config_id                      | architecture   |   n_seeds |   specs_deployed_swa |   val_rmse_live |   val_rmse_swa |   test_r2 |   test_rmse |   test_bias |
|:--------------|:-------------------------------|:---------------|----------:|---------------------:|----------------:|---------------:|----------:|------------:|------------:|
| 2regime_54    | fg_w384x384_d0.3_swa           | fg             |         2 |                    0 |       0.0515398 |      0.0999253 |  0.72938  |   0.0529927 | 0.0188537   |
| 2regime_54    | w512x512x512_d0.3_huber0.1_swa | mlp            |         2 |                    0 |       0.051755  |      0.105121  |  0.764039 |   0.049483  | 0.00869853  |
| 2regime_54    | fg_w512x512_d0.3_huber0.1_swa  | fg             |         2 |                    

## Early-Stopping & SWA-rule Replay (patience-60 re-check)

1.3 confirmed no honest rule beats patience-60 on the plain-MLP curves. 2.0 replays the rules on the new curves including a `swa_val` rule (argmin of the SWA-snapshot val curve, from `curves_swa.npy`), checking whether SWA's smoothing changes the stopping picture. Oracle (argmin test) is the unreachable bound. Backed by `analyze_stopping.py` (tag 20).


In [9]:
if df_stop20_agg is not None:
    print("### Stopping-rule replay on 2.0 curves (mean pooled test RMSE over configs/seeds)")
    print("Lower is better. 'oracle' = argmin on test (reference bound, not achievable honestly).")
    for family, fam_label in fam_labels.items():
        sub = df_stop20_agg[df_stop20_agg["family"] == family].sort_values("mean_test_rmse")
        if sub.empty:
            continue
        print(f"\n#### {fam_label}")
        print(sub[["rule", "mean_test_rmse", "median_test_rmse", "n"]].to_markdown(index=False))
else:
    print("stopping_20_aggregate.csv not found — run analyze_stopping.py after the sweep.")


### Stopping-rule replay on 2.0 curves (mean pooled test RMSE over configs/seeds)
Lower is better. 'oracle' = argmin on test (reference bound, not achievable honestly).

#### 2-Regime-96
| rule             |   mean_test_rmse |   median_test_rmse |   n |
|:-----------------|-----------------:|-------------------:|----:|
| oracle           |        0.0503049 |          0.0479306 |  19 |
| patience60       |        0.0545378 |          0.0522245 |  19 |
| patience40       |        0.0545378 |          0.0522245 |  19 |
| patience20       |        0.0545378 |          0.0522245 |  19 |
| val_aux          |        0.0562256 |          0.0548668 |  19 |
| plateau_w60e1e-4 |        0.0599735 |          0.0555584 |  19 |
| swa_val          |        0.0625803 |          0.0612281 |  19 |
| plateau_w40e1e-4 |        0.0639599 |          0.059757  |  19 |
| plateau_w40e3e-4 |        0.0639599 |          0.059757  |  19 |
| plateau_w20e1e-4 |        0.0733378 |          0.0792595 |  19 |

#### 2-R

## FeatureGroupedMLP — the semantic grouping

The `fg` architecture's tower structure is resolved from the feature names by `mlp20/feature_groups.py` (explicit, first-match-wins rule table; validated: every feature in exactly one group, raises otherwise). This table shows the grouping for the union of features used by the 3 families.


In [10]:
import sys
sys.path.insert(0, str(EXP_DIR))
sys.path.insert(0, str(EXP_DIR.parent / "derived_8.4-eval-1.1"))
import yaml
with open(EXP_DIR / "config.yaml") as f:
    config = yaml.safe_load(f)
from eval11.data import load_experiment_data
from run_mlp_sweep import family_features
from mlp20.feature_groups import group_features
PROJECT_ROOT = EXP_DIR.parents[2]
data = load_experiment_data(PROJECT_ROOT, config)
all_feats = set(config["shared_backbone_54"])
for fam in config["families"]:
    for k, v in family_features(fam["id"], config, data).items():
        all_feats.update(v)
fg = group_features(sorted(all_feats))
print(f"### Feature groups ({fg.n_features} unique features, {fg.n_groups} groups)")
for gid, idxs in enumerate(fg.groups):
    print(f"**group {gid} ({len(idxs)} features):** {', '.join(fg.names[i] for i in idxs)}")


### Feature groups (116 unique features, 8 groups)
**group 0 (26 features):** A_d_SMAP_sm_interp_kobs14, A_d_SMAP_sm_interp_kobs30, A_grad_SMAP_sm_interp_kobs14, A_grad_SMAP_sm_interp_kobs30, A_grad_SMAP_sm_interp_kobs7, C_lag_SMAP_sm_interp_kobs12, C_lag_SMAP_sm_interp_kobs30, SMAP_ampm_diff_interp, SMAP_sm_am_interp, SMAP_sm_am_interp_lag1, SMAP_sm_am_interp_lag30, SMAP_sm_am_interp_rollrange30, SMAP_sm_am_interp_rollrange7, SMAP_sm_interp_lag7, SMAP_sm_interp_rollrange30, SMAP_sm_interp_rollrange7, SMAP_sm_pm_interp, SMAP_sm_pm_interp_lag1, SMAP_sm_pm_interp_lag30, SMAP_sm_pm_interp_lag7, SMAP_sm_pm_interp_rollmean30, SMAP_sm_pm_interp_rollrange30, SMAP_sm_pm_interp_rollrange7, V_ema_SMAP_sm_interp_kobs30, V_rollmin_SMAP_sm_interp_kobs14, V_rollmin_SMAP_sm_interp_kobs30
**group 1 (7 features):** A_grad_s2_b11_kobs30, V_rollmin_s2_b11_kobs14, V_rollmin_s2_b11_kobs30, V_rollmin_s2_b12_kobs30, V_rollrng_s2_b11_kobs30, s2_b4, s2_b8
**group 2 (17 features):** C_lag_F_NDMI_kobs30, C_lag_F

## Extrapolation (OOD) Check

Test rows whose top-10 gain features fall outside the trainval [min, max] range are flagged as OOD (same definition as mlp-1.1/1.2/1.3). Compares the best neural models per family (and 5-seed champion ensembles when present) vs the XGBoost references on in-distribution vs OOD slices.


In [11]:
if df_ood is not None:
    print("### OOD Slice Metrics (best neural model per family vs XGBoost references)")
    print(df_ood[["model", "slice", "n", "r2", "rmse", "bias", "mae"]].to_markdown(index=False))
else:
    print("ood_summary.csv not found — run analyze_extrapolation.py after the sweep.")


### OOD Slice Metrics (best neural model per family vs XGBoost references)
| model                                             | slice           |    n |        r2 |      rmse |        bias |       mae |
|:--------------------------------------------------|:----------------|-----:|----------:|----------:|------------:|----------:|
| MLP 2regime-96 (w512x512x512_d0.3_lr1e-3)         | all             | 6620 | 0.761018  | 0.0497987 |  0.0176019  | 0.0384117 |
| MLP 2regime-96 (w512x512x512_d0.3_lr1e-3)         | in_distribution | 6032 | 0.755427  | 0.0512552 |  0.0204626  | 0.0397959 |
| MLP 2regime-96 (w512x512x512_d0.3_lr1e-3)         | ood             |  588 | 0.750658  | 0.0311459 | -0.0117442  | 0.0242127 |
| MLP 2regime-96 (5-seed champ)                     | all             | 6620 | 0.75656   | 0.050261  |  0.0204007  | 0.0391776 |
| MLP 2regime-96 (5-seed champ)                     | in_distribution | 6032 | 0.749983  | 0.0518224 |  0.0227929  | 0.0407259 |
| MLP 2regime-96 (5-se

## Overfitting-Symptom Analysis

Quantifies the generalization failure modes of the 2.0 sweep from the saved artifacts (no retraining): (1) the **train-fit vs held-out gap** — aux2020 RMSE vs val vs test; (2) **capacity vs transfer** by n_params bucket; (3) residual nets (reference) best in-sample, worst test; (4) the **per-epoch curve shape** of each family's 2-seed val winner; (5) **systematic bias** on test vs XGBoost. Backed by `analyze_overfitting.py`.


In [12]:
import sys
sys.path.insert(0, str(EXP_DIR))
from analyze_overfitting import compute_overfitting, print_report
_overfit = compute_overfitting(df_sweep, EXP_DIR)
print_report(_overfit)


OVERFITTING-SYMPTOM ANALYSIS — derived_8.4-eval-mlp-2.0 (from sweep artifacts)

### 1. Train-fit vs held-out gap (median RMSE over 2-regime MLP configs)
| family     |   aux2020 (train-fit) |   val |   test |   val/train ratio |
|:-----------|----------------------:|------:|-------:|------------------:|
| 2regime_96 | 0.0317 | 0.0519 | 0.0501 | 1.6x |
| 2regime_54 | 0.0275 | 0.0577 | 0.0494 | 2.1x |
| 2regime_mixed | 0.0296 | 0.0523 | 0.0488 | 1.8x |

### 2. Capacity vs test transfer (median by n_params bucket)
| family     | capacity   |   n_configs |   med_val_rmse |   med_test_r2 |   med_test_bias |
|:-----------|:-----------|------------:|---------------:|--------------:|----------------:|
| 2regime_96 | <200k | 2 | 0.0561 | 0.7844 | 0.0072 |
| 2regime_96 | 500k-1M | 2 | 0.0566 | 0.6248 | 0.0234 |
| 2regime_96 | 1M+ | 6 | 0.0500 | 0.7358 | 0.0202 |
| 2regime_54 | 200-500k | 4 | 0.0619 | 0.7809 | 0.0022 |
| 2regime_54 | 1M+ | 6 | 0.0575 | 0.7286 | 0.0189 |
| 2regime_mixed | <200k | 

## Timing

Sweep wall time, per-job GPU-seconds, and the eval wall time from `timing_log.json` / `timing_summary.csv`. Budget: 2.0 H100-hours.


In [13]:
print("### Timing (H100 PCIe 80 GB, 8 parallel workers)")
print(f"Total sweep wall time: {timing_log.get('sweep_wall_s', float('nan')):.1f} s  |  eval wall time: {timing_log.get('eval_wall_s', float('nan')):.1f} s")
print(f"Total training time (all jobs, GPU-seconds): {sum(j.get('train_time_s', 0.0) for j in timing_log.get('jobs', {}).values()):.0f} s "
      f"= {sum(j.get('train_time_s', 0.0) for j in timing_log.get('jobs', {}).values()) / 3600:.2f} GPU-hours (budget: 2.0)")
print("\n### Per-job training time (slowest 10)")
trows = df_timing.sort_values("train_time_s", ascending=False).head(10)
print(trows[["family", "config_id", "architecture", "train_time_s", "epochs", "best_epoch", "val_rmse", "test_r2"]].to_markdown(index=False))


### Timing (H100 PCIe 80 GB, 8 parallel workers)
Total sweep wall time: 798.6 s  |  eval wall time: 8.8 s
Total training time (all jobs, GPU-seconds): 4884 s = 1.36 GPU-hours (budget: 2.0)

### Per-job training time (slowest 10)
| family        | config_id                      | architecture   |   train_time_s |   epochs |   best_epoch |   val_rmse |   test_r2 |
|:--------------|:-------------------------------|:---------------|---------------:|---------:|-------------:|-----------:|----------:|
| 2regime_54    | fg_w384x384_d0.3_swa           | fg             |        444.094 |      400 |          342 |  0.0575753 |  0.72938  |
| 2regime_54    | fg_w384x384_d0.3               | fg             |        419.867 |      400 |          352 |  0.0575515 |  0.727882 |
| 2regime_54    | fg_w512x512_d0.3_huber0.1_swa  | fg             |        366.337 |      344 |          284 |  0.0577613 |  0.721153 |
| 2regime_mixed | fg_w384x384_d0.3_gelu_swa      | fg             |        331.653 |      4